# 05 — Modelagem dimensional e integração da Tríade

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Entrada:** tabelas tratadas de `data/processed/` (`fato_macro`, `fato_cotacoes`, `fato_contabil`).

**Modelo:** esquema estrela simplificado — uma dimensão de empresas (`dim_empresa`) ligada às tabelas de fatos, e uma tabela integrada anual (`fato_anual_integrado`) com uma linha por empresa e exercício.

**Saída:** `dim_empresa.csv`, `fato_indicadores_contabeis.csv`, `fato_macro_anual.csv`, `fato_mercado_anual.csv` e `fato_anual_integrado.csv` em `data/processed/`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 100)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"

macro = pd.read_csv(PROC_DIR / "fato_macro.csv", parse_dates=["data"])
cotacoes = pd.read_csv(PROC_DIR / "fato_cotacoes.csv", parse_dates=["data"])
contabil = pd.read_csv(PROC_DIR / "fato_contabil.csv", dtype={"cnpj": str, "cd_conta": str})

print("macro:", macro.shape, "| cotacoes:", cotacoes.shape, "| contabil:", contabil.shape)

macro: (3201, 3) | cotacoes: (11169, 9) | contabil: (9894, 8)


In [2]:
CARTEIRA = {
    "LREN3": ("Lojas Renner",  "Consumo/Varejo"),
    "ABEV3": ("Ambev",         "Consumo/Varejo"),
    "PETR4": ("Petrobras",     "Energia"),
    "EGIE3": ("Engie Brasil",  "Energia"),
    "VALE3": ("Vale",          "Commodities Exportadoras"),
    "SUZB3": ("Suzano",        "Commodities Exportadoras"),
    "WEGE3": ("WEG",           "Exportadoras Industriais"),
    "EMBR3": ("Embraer",       "Exportadoras Industriais"),
    "ITUB4": ("Itaú Unibanco", "Bancos"),
}

dim_empresa = pd.DataFrame(
    [(ticker, nome, setor) for ticker, (nome, setor) in CARTEIRA.items()],
    columns=["ticker", "empresa", "setor"],
)
dim_empresa["tipo"] = dim_empresa["setor"].map(lambda s: "Financeira" if s == "Bancos" else "Não financeira")
dim_empresa = dim_empresa.merge(contabil[["ticker", "cnpj"]].drop_duplicates(), on="ticker", how="left")

dim_empresa.to_csv(PROC_DIR / "dim_empresa.csv", index=False)
dim_empresa

,ticker,empresa,setor,tipo,cnpj
0,LREN3,Lojas Renner,Consumo/Varejo,Não financeira,92.754.738/0001-62
1,ABEV3,Ambev,Consumo/Varejo,Não financeira,07.526.557/0001-00
2,PETR4,Petrobras,Energia,Não financeira,33.000.167/0001-01
3,EGIE3,Engie Brasil,Energia,Não financeira,02.474.103/0001-19
4,VALE3,Vale,Commodities Exportadoras,Não financeira,33.592.510/0001-54
5,SUZB3,Suzano,Commodities Exportadoras,Não financeira,16.404.287/0001-55
6,WEGE3,WEG,Exportadoras Industriais,Não financeira,84.429.695/0001-11
7,EMBR3,Embraer,Exportadoras Industriais,Não financeira,07.689.002/0001-89
8,ITUB4,Itaú Unibanco,Bancos,Financeira,60.872.504/0001-23


In [3]:
CONTAS = {
    "receita_liquida": "3.01",
    "resultado_bruto": "3.03",
    "ebit": "3.05",
    "despesas_financeiras": "3.06.02",
    "lucro_liquido": "3.11",
    "ativo_total": "1",
    "ativo_circulante": "1.01",
    "caixa": "1.01.01",
    "aplicacoes_financeiras": "1.01.02",
    "passivo_circulante": "2.01",
    "emprestimos_cp": "2.01.04",
    "emprestimos_lp": "2.02.01",
    "patrimonio_liquido": "2.03",
}

nao_fin = dim_empresa.loc[dim_empresa["tipo"] == "Não financeira", "ticker"]
recorte = contabil[(contabil["ano"] == 2023)
                   & contabil["ticker"].isin(nao_fin)
                   & contabil["cd_conta"].isin(CONTAS.values())]

(recorte.groupby(["cd_conta", "ds_conta"])["ticker"]
        .apply(lambda s: ", ".join(sorted(s)))
        .reset_index(name="empresas"))

,cd_conta,ds_conta,empresas
0,1,Ativo Total,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
1,1.01,Ativo Circulante,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
2,1.01.01,Caixa e Equivalentes de Caixa,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
3,1.01.02,Aplicações Financeiras,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
4,2.01,Passivo Circulante,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
5,2.01.04,Empréstimos e Financiamentos,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
6,2.02.01,Empréstimos e Financiamentos,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
7,2.03,Patrimônio Líquido Consolidado,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
8,3.01,Receita de Venda de Bens e/ou Serviços,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
9,3.03,Resultado Bruto,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."


In [4]:
itau = contabil[(contabil["ticker"] == "ITUB4")
                & (contabil["ano"] == 2023)
                & (contabil["cd_conta"].str.count(r"\.") <= 1)]
itau[["demonstrativo", "cd_conta", "ds_conta", "valor_reais"]]

,demonstrativo,cd_conta,ds_conta,valor_reais
4195,BPA,1,Ativo Total,2.543100e+12
4196,BPA,1.01,Caixa e Equivalentes de Caixa,3.200100e+10
4197,BPA,1.02,Ativos Financeiros,2.384618e+12
4216,BPA,1.03,Tributos Diferidos,6.452100e+10
4220,BPA,1.04,Outros Ativos,2.016800e+10
4225,BPA,1.05,Investimentos,9.293000e+09
4230,BPA,1.06,Imobilizado,9.135000e+09
4233,BPA,1.07,Intangível,2.336400e+10
4236,BPP,2,Passivo Total,2.543100e+12
4237,BPP,2.01,Passivos Financeiros ao Valor Justo através do...,5.333100e+10


## Etapa 1 — Conclusão: dimensão e plano de contas

- `dim_empresa`: 9 companhias, 5 setores, classificadas em **Não financeira** (8) e **Financeira** (Itaú).
- Os 13 códigos contábeis usados nas empresas não financeiras têm **descrição única e idêntica** em todas elas (plano de contas padronizado da CVM).
- O **Itaú** segue o plano de contas de instituição financeira: Lucro Líquido = **3.09**, Patrimônio Líquido = **2.08**, Receita = **3.01** (intermediação financeira). Não há EBIT, Ativo/Passivo Circulante nem Dívida Líquida comparáveis.
- **Decisão:** para o Itaú são calculados apenas Receita, Lucro, PL e ROE; os demais indicadores ficam como não aplicáveis (N/A).